<a href="https://colab.research.google.com/github/a10ns0/Test_Modelos_Cuantizacion/blob/main/TrainingKeywordSpotting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training Keyword Spotting
This notebook builds on the Colab in which we used the pre-trained [micro_speech](https://github.com/tensorflow/tensorflow/tree/v2.4.1/tensorflow/lite/micro/examples/micro_speech) example.

**It is now time for you to train your own custom keyword spotting model using the Speech Commands dataset!**

#Setup

## Import packages
Clone the TensorFlow Github Repository, which contains the relevant code required to run this assignment.

In [1]:
import os
import subprocess
import sys

# Use Keras 2 (the API the course code was written against)
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# IMPORTANT: do NOT upgrade protobuf in this notebook.
# This notebook trains via the legacy speech_commands train.py, which calls
# tf.io.write_graph -> protobuf text_format.MessageToString(float_format=...).
# Newer protobuf (>=6.31) removed that kwarg, which breaks training. This
# notebook does not use tensorflow_datasets, so keep Colab's default protobuf.
try:
    import tensorflow as tf
    import tf_keras
    import tensorflow_hub as hub
    print(f"TensorFlow {tf.__version__} (tf_keras {tf_keras.__version__})")
    print(f"tensorflow_hub {hub.__version__}")
except ImportError:
    print("Installing required packages...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "tensorflow", "tf_keras", "tensorflow_hub"])
    print()
    print("=" * 60)
    print("Install complete. Now: Runtime > Restart session, then re-run this cell.")
    print("=" * 60)


TensorFlow 2.20.0 (tf_keras 2.20.0)
tensorflow_hub 0.16.1


In [ ]:
!wget https://github.com/tensorflow/tensorflow/archive/v2.4.1.zip
!unzip v2.4.1.zip &> 0
!mv tensorflow-2.4.1/ tensorflow/

In [ ]:
import tensorflow.compat.v1 as tf
#assert(tf.VERSION == "1.15.0") #Make sure you install Tensorflow 1.15.0 and restart the runtime or this will fail!
# Now that the runtime is set import things!
import sys
# We add this path so we can import the speech processing modules.
sys.path.append("/content/tensorflow/tensorflow/examples/speech_commands/")
import input_data
import models
import numpy as np
import pickle

## Configure Your Model!
Select your keywords and model settings with which to train!

**This is where you need to make choices and input data!**

```WANTED_WORDS``` = A comma-delimited string of the words you want to train for (e.g., "yes,no"). All the other words you do not select will be used to train an "unknown" label so that the model does not just recognize speech but your specific words. Audio data with no spoken words will be used to train a "silence" label. We suggest picking 2-4 words for best results.

Options for target words are (PICK FROM THIS LIST FOR BEST RESULTS): "one", "two", "three", "four", "five", "six", "seven", "eight", "nine",  "yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go", “backward”, “forward”, “follow”, “learn”,

Additional words that will be used to help train the "unkown" label are: "bed", "bird", "cat", "dog", "happy", "house", "marvin", "sheila", "tree", "wow"

In [ ]:
WANTED_WORDS = "yes,no,stop,one,two,three"

The number of training steps and learning rates can be specified as comma-separated strings to define the amount/rate at each stage. For example, ```TRAINING_STEPS="12000,3000"``` and ```LEARNING_RATE="0.001,0.0001"``` will run 12,000 training steps with a rate of 0.001 followed by 3,000 final steps with a learning rate of 0.0001. These are good default values to work off of when you choose your values as the course staff has gotten this to work well with those values in the past!

In [ ]:
TRAINING_STEPS = "3000,1000"
LEARNING_RATE = "0.001,0.0001"

We suggest you leave the ```MODEL_ARCHITECTURE``` as tiny_conv the first time but if you would like to do this again and explore additional models some options are: ```single_fc, conv, low_latency_conv, low_latency_svdf, tiny_embedding_conv```

In [ ]:
MODEL_ARCHITECTURE = 'tiny_conv'

In [ ]:
# Calculate the total number of steps, which is used to identify the checkpoint
# file name.
TOTAL_STEPS = str(sum(map(lambda string: int(string), TRAINING_STEPS.split(","))))

# Print the configuration to confirm it
print("Training these words: %s" % WANTED_WORDS)
print("Training steps in each stage: %s" % TRAINING_STEPS)
print("Learning rate in each stage: %s" % LEARNING_RATE)
print("Total number of training steps: %s" % TOTAL_STEPS)

**DO NOT MODIFY** the following constants as they include filepaths used in this notebook and data that is shared during training and inference.

In [ ]:
# Calculate the percentage of 'silence' and 'unknown' training samples required
# to ensure that we have equal number of samples for each label.
number_of_labels = WANTED_WORDS.count(',') + 1
number_of_total_labels = number_of_labels + 2 # for 'silence' and 'unknown' label
equal_percentage_of_training_samples = int(100.0/(number_of_total_labels))
SILENT_PERCENTAGE = equal_percentage_of_training_samples
UNKNOWN_PERCENTAGE = equal_percentage_of_training_samples

# Constants which are shared during training and inference
PREPROCESS = 'micro'
WINDOW_STRIDE = 20

# Constants used during training only
VERBOSITY = 'DEBUG'
EVAL_STEP_INTERVAL = "250"
SAVE_STEP_INTERVAL = "250"

# Constants for training directories and filepaths
DATASET_DIR =  'dataset/'
LOGS_DIR = 'logs/'
TRAIN_DIR = 'train/' # for training checkpoints and other files.

# Constants for inference directories and filepaths
import os
MODELS_DIR = 'models'
if not os.path.exists(MODELS_DIR):
  os.mkdir(MODELS_DIR)
MODEL_TF = os.path.join(MODELS_DIR, 'model.pb')
MODEL_TFLITE = os.path.join(MODELS_DIR, 'model.tflite')
PTQ_INT8_TFLITE = os.path.join(MODELS_DIR, 'tinyconv_ptq_int8.tflite')
PTQ_INT16X8_TFLITE = os.path.join(MODELS_DIR, 'tinyconv_ptq_int16x8.tflite')
QAT_INT8_TFLITE = os.path.join(MODELS_DIR, 'tinyconv_qat_int8.tflite')
QAT_SAVED_MODEL = os.path.join(MODELS_DIR, 'saved_model_qat')
QAT_TRAIN_DIR = 'train_qat/'
FLOAT_MODEL_TFLITE = os.path.join(MODELS_DIR, 'float_model.tflite')
MODEL_TFLITE_MICRO = os.path.join(MODELS_DIR, 'model.cc')
SAVED_MODEL = os.path.join(MODELS_DIR, 'saved_model')

# Constants for Quantization
QUANT_INPUT_MIN = 0.0
QUANT_INPUT_MAX = 26.0
QUANT_INPUT_RANGE = QUANT_INPUT_MAX - QUANT_INPUT_MIN

# Constants for audio process during Quantization and Evaluation
SAMPLE_RATE = 16000
CLIP_DURATION_MS = 1000
WINDOW_SIZE_MS = 30.0
FEATURE_BIN_COUNT = 40
BACKGROUND_FREQUENCY = 0.8
BACKGROUND_VOLUME_RANGE = 0.1
TIME_SHIFT_MS = 100.0

# URL for the dataset and train/val/test split
DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz'
VALIDATION_PERCENTAGE = 10
TESTING_PERCENTAGE = 10

In [ ]:
# Calculate the correct flattened input data shape for later use in model conversion
# since the model takes a flattened version of the spectrogram. The shape is number of
# overlapping windows times the number of frequency bins. For the default settings we have
# 40 bins (as set above) times 49 windows (as calculated below) so the shape is (1,1960)
def window_counter(total_samples, window_size, stride):
  '''helper function to count the number of full-length overlapping windows'''
  window_count = 0
  sample_index = 0
  while True:
    window = range(sample_index,sample_index+stride)
    if window.stop < total_samples:
      window_count += 1
    else:
      break

    sample_index += stride
  return window_count

OVERLAPPING_WINDOWS = window_counter(CLIP_DURATION_MS, int(WINDOW_SIZE_MS), WINDOW_STRIDE)
FLATTENED_SPECTROGRAM_SHAPE = (1, OVERLAPPING_WINDOWS * FEATURE_BIN_COUNT)

# Train the model

### Load in TensorBoard to visulaize the training process.

As training progresses you should see the training status show up in the Tensorboard area. If this works it is very helpful for analyzing your training progress. Unfortunately, the staff has found that it sometimes doesn't start showing data for a while (~15 minutes) and sometimes doesn't show data until training completes (and instead shows ```No dashboards are active for the current data set```.). If it is working and then stops updating look to the top of the cell and click reconnect.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOGS_DIR}

### Launch Training

**ARE YOU USING A GPU INSTANCE!?!?!**
Make sure to check ```runtime->change runtime type``` as a GPU runtime will take ~2 hours to complete training whereas a CPU runtime with take ~10 hours!

**Importantly, every hour or so you should make sure to remind colab you are still there by clicking around in a cell or clicking reconnect on the tensorboard cell (after 90 minutes of no activity Colab may kill the instance)!** But/and beyond that you can absolutely minimize the window and continue with other work.

**If you run training a second time** we suggest restarting your runtime ```runtime->restart runtime``` in order to ensure all caches are cleared. However, do note that doing so will clear all cells and so you will have to run the entire file again.

If you would like to get more information on the training script you can find the source code for the script [here](https://github.com/tensorflow/tensorflow/blob/master/tensorflow/examples/speech_commands/train.py). In short it sets up the optimizer and preprocessor based on all of the flags we pass in!

Finally, by setting the ```VERBOSITY = 'DEBUG'``` above be aware that the training cell will print A LOT of information. Specifically you will get the accuracy and loss at each step as well as a confusion matrix every 1000 steps. We hope that is helpful in case TensorBoard fails to work. If you would like to run with less printouts you can change the setting to ```WARN``` or ```FATAL```. You will find this in the "Configure Your Model!" section.

In [ ]:
!python tensorflow/tensorflow/examples/speech_commands/train.py \
--data_dir={DATASET_DIR} \
--wanted_words={WANTED_WORDS} \
--silence_percentage={SILENT_PERCENTAGE} \
--unknown_percentage={UNKNOWN_PERCENTAGE} \
--preprocess={PREPROCESS} \
--window_stride={WINDOW_STRIDE} \
--model_architecture={MODEL_ARCHITECTURE} \
--how_many_training_steps={TRAINING_STEPS} \
--learning_rate={LEARNING_RATE} \
--train_dir={TRAIN_DIR} \
--summaries_dir={LOGS_DIR} \
--verbosity={VERBOSITY} \
--eval_step_interval={EVAL_STEP_INTERVAL} \
--save_step_interval={SAVE_STEP_INTERVAL}

# Generating your Model
Just like with the pre-trained model we will now take the final checkpoint and convert it into a quantized TensorFlow Lite model.

### Generate a TensorFlow Model for Inference

Combine relevant training results (graph, weights, etc) into a single file for inference. This process is known as freezing a model and the resulting model is known as a frozen model/graph, as it cannot be further re-trained after this process.

In [ ]:
# These commands modify the legacy TensorFlow example code to make it compatible with TensorFlow 2.x by replacing deprecated imports and removing
# runtime placeholders with constants, ensuring the model can be frozen and converted to TFLite without requiring dynamic inputs.

# Patch freeze.py to use the TF 2.x-compatible version of convert_variables_to_constants from tf.compat.v1.
!sed -i 's/from tensorflow\.python\.framework import graph_util/from tensorflow.compat.v1.graph_util import convert_variables_to_constants/' tensorflow/tensorflow/examples/speech_commands/freeze.py

# Update function call to match the new import (remove graph_util prefix).
!sed -i 's/graph_util\.convert_variables_to_constants/convert_variables_to_constants/' tensorflow/tensorflow/examples/speech_commands/freeze.py

# Replace placeholder for background_volume with a constant to eliminate runtime input requirements.
!sed -i "s/tf\.compat\.v1\.placeholder(tf\.float32, name='background_volume')/tf.constant(0.0, dtype=tf.float32, name='background_volume')/" tensorflow/tensorflow/examples/speech_commands/input_data.py

# Replace placeholder for time_shift_padding with a constant to avoid dynamic input during inference.
!sed -i "s/tf\.compat\.v1\.placeholder(tf\.int32, shape=\[2,2\], name='time_shift_padding')/tf.constant([[0, 0], [0, 0]], dtype=tf.int32, name='time_shift_padding')/" tensorflow/tensorflow/examples/speech_commands/input_data.py

# Replace placeholder for foreground_volume with a constant to ensure static graph for TFLite conversion.
!sed -i "s/tf\.compat\.v1\.placeholder(tf\.float32, name='foreground_volume')/tf.constant(1.0, dtype=tf.float32, name='foreground_volume')/" tensorflow/tensorflow/examples/speech_commands/input_data.py


In [ ]:
!rm -rf {SAVED_MODEL}
!python tensorflow/tensorflow/examples/speech_commands/freeze.py \
--wanted_words=$WANTED_WORDS \
--window_stride_ms=$WINDOW_STRIDE \
--preprocess=$PREPROCESS \
--model_architecture=$MODEL_ARCHITECTURE \
--start_checkpoint=$TRAIN_DIR$MODEL_ARCHITECTURE'.ckpt-'{TOTAL_STEPS} \
--save_format=saved_model \
--output_file={SAVED_MODEL}

In [ ]:
float_converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL)
float_tflite_model = float_converter.convert()

with open(FLOAT_MODEL_TFLITE, "wb") as f:
    f.write(float_tflite_model)

### Generate a TensorFlow Lite Model

Convert the frozen graph into a TensorFlow Lite model, which is fully quantized for use with embedded devices.

The following cell will also print the model size, which will be under 20 kilobytes.

We download the dataset to use as a representative dataset for more thoughtful post training quantization.

In [ ]:
model_settings = models.prepare_model_settings(
    len(input_data.prepare_words_list(WANTED_WORDS.split(','))),
    SAMPLE_RATE, CLIP_DURATION_MS, WINDOW_SIZE_MS,
    WINDOW_STRIDE, FEATURE_BIN_COUNT, PREPROCESS)
audio_processor = input_data.AudioProcessor(
    DATA_URL, DATASET_DIR,
    SILENT_PERCENTAGE, UNKNOWN_PERCENTAGE,
    WANTED_WORDS.split(','), VALIDATION_PERCENTAGE,
    TESTING_PERCENTAGE, model_settings, LOGS_DIR)

In [ ]:
with tf.compat.v1.Session() as sess:

    converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    def representative_dataset_gen():
        for i in range(100):
            data, _ = audio_processor.get_data(
                1, i, model_settings,
                BACKGROUND_FREQUENCY,
                BACKGROUND_VOLUME_RANGE,
                TIME_SHIFT_MS,
                'testing',
                sess
            )
            flattened = np.array(data.flatten(), dtype=np.float32).reshape(FLATTENED_SPECTROGRAM_SHAPE)
            yield [flattened]

    converter.representative_dataset = representative_dataset_gen


    quantized_tflite_model = converter.convert()

    # Save to file
    with open(PTQ_INT8_TFLITE, "wb") as f:
        f.write(quantized_tflite_model)

    print(f"PTQ INT8 guardado en {PTQ_INT8_TFLITE}, tamaño: {len(quantized_tflite_model)} bytes")


### Testing the accuracy after Quantization

Verify that the model we've exported is still accurate, using the TF Lite Python API and our test set.

In [ ]:
# Helper function to run inference
def run_tflite_inference_testSet(tflite_model_path, model_type="Float"):
  #
  # Load test data
  #
  np.random.seed(0) # set random seed for reproducible test results.
  with tf.Session() as sess:
  # with tf.compat.v1.Session() as sess: #replaces the above line for use with TF2.x
    test_data, test_labels = audio_processor.get_data(
        -1, 0, model_settings, BACKGROUND_FREQUENCY, BACKGROUND_VOLUME_RANGE,
        TIME_SHIFT_MS, 'testing', sess)
  test_data = np.expand_dims(test_data, axis=1).astype(np.float32)

  #
  # Initialize the interpreter
  #
  interpreter = tf.lite.Interpreter(tflite_model_path)
  interpreter.allocate_tensors()
  input_details = interpreter.get_input_details()[0]
  output_details = interpreter.get_output_details()[0]

  #
  # For quantized models, manually quantize the input data from float to integer
  #
  if model_type == "Quantized":
    input_scale, input_zero_point = input_details["quantization"]
    test_data = test_data / input_scale + input_zero_point
    test_data = test_data.astype(input_details["dtype"])

  #
  # Evaluate the predictions
  #
  correct_predictions = 0
  for i in range(len(test_data)):
    interpreter.set_tensor(input_details["index"], test_data[i])
    interpreter.invoke()
    output = interpreter.get_tensor(output_details["index"])[0]
    top_prediction = output.argmax()
    correct_predictions += (top_prediction == test_labels[i])

  print('%s model accuracy is %f%% (Number of test samples=%d)' % (
      model_type, (correct_predictions * 100) / len(test_data), len(test_data)))

In [ ]:
# Compute float model accuracy
run_tflite_inference_testSet(FLOAT_MODEL_TFLITE)

# Compute quantized model accuracy
run_tflite_inference_testSet(MODEL_TFLITE, model_type='Quantized')

# Testing the model with your own data!
Now comes the fun part. It's time to test your model with your own realworld data. We'll proceed in the same way we tested the pre-trained model. Have fun!

### Importing packages

In [ ]:
!pip install ffmpeg-python &> 0
from IPython.display import HTML, Audio
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
from scipy.io.wavfile import read as wav_read
import io
import ffmpeg
!pip install librosa
import librosa
import scipy.io.wavfile
!git clone https://github.com/petewarden/extract_loudest_section.git
!make -C extract_loudest_section/
print("Packages Imported, Extract_Loudest_Section Built")

### Define the helper function to run inference

In [ ]:
# Helper function to run inference (on a single input this time)
# Note: this also includes additional manual pre-processing
TF_SESS = tf.compat.v1.InteractiveSession()
def run_tflite_inference_singleFile(tflite_model_path, custom_audio, sr_custom_audio, model_type="Float"):
  #
  # Preprocess the sample to get the features we pass to the model
  #
  # First re-sample to the needed rate (and convert to mono if needed)
  custom_audio_resampled = librosa.resample(y = librosa.to_mono(np.float64(custom_audio)), orig_sr = sr_custom_audio, target_sr = SAMPLE_RATE)
  # Then extract the loudest one second
  scipy.io.wavfile.write('custom_audio.wav', SAMPLE_RATE, np.int16(custom_audio_resampled))
  !/tmp/extract_loudest_section/gen/bin/extract_loudest_section custom_audio.wav ./trimmed
  # Finally pass it through the TFLiteMicro preprocessor to produce the
  # spectrogram/MFCC input that the model expects
  custom_model_settings = models.prepare_model_settings(
      0, SAMPLE_RATE, CLIP_DURATION_MS, WINDOW_SIZE_MS,
      WINDOW_STRIDE, FEATURE_BIN_COUNT, PREPROCESS)
  custom_audio_processor = input_data.AudioProcessor(None, None, 0, 0, '', 0, 0,
                                                    model_settings, None)
  custom_audio_preprocessed = custom_audio_processor.get_features_for_wav(
                                        'trimmed/custom_audio.wav', model_settings, TF_SESS)
  # Reshape the output into a 1,1960 matrix as that is what the model expects
  custom_audio_input = custom_audio_preprocessed[0].flatten()
  test_data = np.reshape(custom_audio_input,(1,len(custom_audio_input)))

  #
  # Initialize the interpreter
  #
  interpreter = tf.lite.Interpreter(tflite_model_path)
  interpreter.allocate_tensors()
  input_details = interpreter.get_input_details()[0]
  output_details = interpreter.get_output_details()[0]

  #
  # For quantized models, manually quantize the input data from float to integer
  #
  if model_type == "Quantized":
    input_scale, input_zero_point = input_details["quantization"]
    test_data = test_data / input_scale + input_zero_point
    test_data = test_data.astype(input_details["dtype"])

  #
  # Run the interpreter
  #
  interpreter.set_tensor(input_details["index"], test_data)
  interpreter.invoke()
  output = interpreter.get_tensor(output_details["index"])[0]
  top_prediction = output.argmax()

  #
  # Translate the output
  #
  top_prediction_str = ''
  if top_prediction >= 2:
    top_prediction_str = WANTED_WORDS.split(',')[top_prediction-2]
  elif top_prediction == 0:
    top_prediction_str = 'silence'
  else:
    top_prediction_str = 'unknown'

  print('%s model guessed the value to be %s' % (model_type, top_prediction_str))

### Define the audio importing function
Adapted from: https://ricardodeazambuja.com/deep_learning/2019/03/09/audio_and_video_google_colab/ and https://colab.research.google.com/drive/1Z6VIRZ_sX314hyev3Gm5gBqvm1wQVo-a#scrollTo=RtMcXr3o6gxN

In [ ]:
AUDIO_HTML = """
<script>
var my_div = document.createElement("DIV");
var my_p = document.createElement("P");
var my_btn = document.createElement("BUTTON");
var t = document.createTextNode("Press to start recording");

my_btn.appendChild(t);
//my_p.appendChild(my_btn);
my_div.appendChild(my_btn);
document.body.appendChild(my_div);

var base64data = 0;
var reader;
var recorder, gumStream;
var recordButton = my_btn;

var handleSuccess = function(stream) {
  gumStream = stream;
  var options = {
    //bitsPerSecond: 8000, //chrome seems to ignore, always 48k
    mimeType : 'audio/webm;codecs=opus'
    //mimeType : 'audio/webm;codecs=pcm'
  };
  //recorder = new MediaRecorder(stream, options);
  recorder = new MediaRecorder(stream);
  recorder.ondataavailable = function(e) {
    var url = URL.createObjectURL(e.data);
    var preview = document.createElement('audio');
    preview.controls = true;
    preview.src = url;
    document.body.appendChild(preview);

    reader = new FileReader();
    reader.readAsDataURL(e.data);
    reader.onloadend = function() {
      base64data = reader.result;
      //console.log("Inside FileReader:" + base64data);
    }
  };
  recorder.start();
  };

recordButton.innerText = "Recording... press to stop";

navigator.mediaDevices.getUserMedia({audio: true}).then(handleSuccess);


function toggleRecording() {
  if (recorder && recorder.state == "recording") {
      recorder.stop();
      gumStream.getAudioTracks()[0].stop();
      recordButton.innerText = "Saving the recording... pls wait!"
  }
}

// https://stackoverflow.com/a/951057
function sleep(ms) {
  return new Promise(resolve => setTimeout(resolve, ms));
}

var data = new Promise(resolve=>{
//recordButton.addEventListener("click", toggleRecording);
recordButton.onclick = ()=>{
toggleRecording()

sleep(2000).then(() => {
  // wait 2000ms for the data to be available...
  // ideally this should use something like await...
  //console.log("Inside data:" + base64data)
  resolve(base64data.toString())

});

}
});

</script>
"""

def get_audio():
  display(HTML(AUDIO_HTML))
  data = eval_js("data")
  binary = b64decode(data.split(',')[1])

  process = (ffmpeg
    .input('pipe:0')
    .output('pipe:1', format='wav', ac='1')
    .run_async(pipe_stdin=True, pipe_stdout=True, pipe_stderr=True, quiet=True, overwrite_output=True)
  )
  output, err = process.communicate(input=binary)

  riff_chunk_size = len(output) - 8
  # Break up the chunk size into four bytes, held in b.
  q = riff_chunk_size
  b = []
  for i in range(4):
      q, r = divmod(q, 256)
      b.append(r)

  # Replace bytes 4:8 in proc.stdout with the actual size of the RIFF chunk.
  riff = output[:4] + bytes(b) + output[8:]

  sr, audio = wav_read(io.BytesIO(riff))

  return audio, sr
print("Chrome Audio Recorder Defined")

### Record your own audio and test the model!
After you run the record cell wait for the stop button to appear then start recording and then press the button to stop the recording once you have said the word!

In [ ]:
custom_audio, sr_custom_audio = get_audio()
print("DONE")

In [ ]:
# Then test the model
run_tflite_inference_singleFile(MODEL_TFLITE, custom_audio, sr_custom_audio, model_type="Quantized")

# Comparación final: Float32, PTQ INT8, QAT INT8 y PTQ INT16x8

Las siguientes celdas amplían el notebook original sin cambiar el dataset ni la
arquitectura `tiny_conv`.

Se comparan cuatro variantes:

1. **Float32**: modelo original.
2. **PTQ INT8**: pesos y activaciones en 8 bits.
3. **QAT INT8**: ajuste consciente de cuantización desde el checkpoint Float32.
4. **PTQ INT16x8**: pesos en 8 bits y activaciones en 16 bits.

La comparación utiliza exactamente el mismo conjunto de prueba para los cuatro
modelos.

## A. Entrenamiento QAT reducido

El ejemplo legado de `speech_commands` incluye una opción `--quantize` que inserta
operaciones de fake quantization durante el entrenamiento. Para ahorrar tiempo, QAT
parte desde el checkpoint Float32 ya entrenado y realiza solamente **1.000 pasos**
adicionales con una tasa de aprendizaje baja.

In [ ]:
# Verificar que la versión descargada del script ofrece la opción QAT.
!python tensorflow/tensorflow/examples/speech_commands/train.py --help | grep -i quantize

In [ ]:
QAT_EXTRA_STEPS = "1000"
QAT_LEARNING_RATE = "0.00005"
QAT_FINAL_STEP = str(int(TOTAL_STEPS) + int(QAT_EXTRA_STEPS))

FLOAT_CHECKPOINT = (
    TRAIN_DIR + MODEL_ARCHITECTURE + ".ckpt-" + TOTAL_STEPS
)

print("Checkpoint Float32 inicial:", FLOAT_CHECKPOINT)
print("Pasos adicionales QAT:", QAT_EXTRA_STEPS)
print("Checkpoint QAT final esperado:", QAT_FINAL_STEP)

In [ ]:
# Ajuste QAT desde el checkpoint Float32.
!rm -rf {QAT_TRAIN_DIR}

!python tensorflow/tensorflow/examples/speech_commands/train.py \
--data_dir={DATASET_DIR} \
--wanted_words={WANTED_WORDS} \
--silence_percentage={SILENT_PERCENTAGE} \
--unknown_percentage={UNKNOWN_PERCENTAGE} \
--preprocess={PREPROCESS} \
--window_stride={WINDOW_STRIDE} \
--model_architecture={MODEL_ARCHITECTURE} \
--how_many_training_steps={QAT_EXTRA_STEPS} \
--learning_rate={QAT_LEARNING_RATE} \
--start_checkpoint={FLOAT_CHECKPOINT} \
--train_dir={QAT_TRAIN_DIR} \
--summaries_dir=logs_qat/ \
--verbosity={VERBOSITY} \
--eval_step_interval=250 \
--save_step_interval=250 \
--quantize=1

In [ ]:
# Congelar el modelo entrenado con fake quantization.
!rm -rf {QAT_SAVED_MODEL}

!python tensorflow/tensorflow/examples/speech_commands/freeze.py \
--wanted_words=$WANTED_WORDS \
--window_stride_ms=$WINDOW_STRIDE \
--preprocess=$PREPROCESS \
--model_architecture=$MODEL_ARCHITECTURE \
--start_checkpoint=$QAT_TRAIN_DIR$MODEL_ARCHITECTURE'.ckpt-'{QAT_FINAL_STEP} \
--save_format=saved_model \
--output_file={QAT_SAVED_MODEL} \
--quantize=1

## B. Conversión del modelo QAT a INT8

Aunque el grafo fue entrenado con fake quantization, se configura explícitamente la
entrada y salida del archivo TFLite como `int8` para mantener la misma interfaz que
el modelo PTQ INT8.

In [ ]:
with tf.compat.v1.Session() as sess:

    def representative_dataset_qat():
        for i in range(200):
            data, _ = audio_processor.get_data(
                1, i, model_settings,
                BACKGROUND_FREQUENCY,
                BACKGROUND_VOLUME_RANGE,
                TIME_SHIFT_MS,
                'training',
                sess
            )
            flattened = np.asarray(
                data.flatten(), dtype=np.float32
            ).reshape(FLATTENED_SPECTROGRAM_SHAPE)
            yield [flattened]

    converter_qat = tf.lite.TFLiteConverter.from_saved_model(QAT_SAVED_MODEL)
    converter_qat.optimizations = [tf.lite.Optimize.DEFAULT]
    converter_qat.representative_dataset = representative_dataset_qat
    converter_qat.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS_INT8
    ]
    converter_qat.inference_input_type = tf.int8
    converter_qat.inference_output_type = tf.int8

    qat_int8_model = converter_qat.convert()

with open(QAT_INT8_TFLITE, "wb") as f:
    f.write(qat_int8_model)

print(
    f"QAT INT8 guardado en {QAT_INT8_TFLITE}: "
    f"{len(qat_int8_model):,} bytes"
)

## C. Conversión PTQ INT16x8

Esta conversión conserva pesos cuantizados a INT8 y utiliza activaciones INT16 en
los operadores compatibles. Se genera directamente desde el mismo SavedModel
Float32 usado para PTQ INT8.

In [ ]:
with tf.compat.v1.Session() as sess:

    def representative_dataset_int16x8():
        for i in range(200):
            data, _ = audio_processor.get_data(
                1, i, model_settings,
                BACKGROUND_FREQUENCY,
                BACKGROUND_VOLUME_RANGE,
                TIME_SHIFT_MS,
                'training',
                sess
            )
            flattened = np.asarray(
                data.flatten(), dtype=np.float32
            ).reshape(FLATTENED_SPECTROGRAM_SHAPE)
            yield [flattened]

    converter_16x8 = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL)
    converter_16x8.optimizations = [tf.lite.Optimize.DEFAULT]
    converter_16x8.representative_dataset = representative_dataset_int16x8

    int16x8_ops = getattr(
        tf.lite.OpsSet,
        "EXPERIMENTAL_TFLITE_BUILTINS_ACTIVATIONS_INT16_WEIGHTS_INT8",
        None
    )

    if int16x8_ops is None:
        raise RuntimeError(
            "Esta versión de TensorFlow no expone "
            "EXPERIMENTAL_TFLITE_BUILTINS_ACTIVATIONS_INT16_WEIGHTS_INT8."
        )

    converter_16x8.target_spec.supported_ops = [int16x8_ops]
    converter_16x8.inference_input_type = tf.int16
    converter_16x8.inference_output_type = tf.int16

    int16x8_model = converter_16x8.convert()

with open(PTQ_INT16X8_TFLITE, "wb") as f:
    f.write(int16x8_model)

print(
    f"PTQ INT16x8 guardado en {PTQ_INT16X8_TFLITE}: "
    f"{len(int16x8_model):,} bytes"
)

## D. Carga única del conjunto de prueba

Para que la comparación sea válida, el conjunto de prueba se carga una sola vez y
se reutiliza sin modificaciones para los cuatro modelos.

In [ ]:
import time
import gzip
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

np.random.seed(0)

with tf.compat.v1.Session() as sess:
    X_test, y_test = audio_processor.get_data(
        -1, 0, model_settings,
        BACKGROUND_FREQUENCY,
        BACKGROUND_VOLUME_RANGE,
        TIME_SHIFT_MS,
        'testing',
        sess
    )

X_test = np.asarray(X_test, dtype=np.float32)
y_test = np.asarray(y_test, dtype=np.int32)

# El intérprete espera un batch de tamaño 1.
X_test = np.expand_dims(X_test, axis=1)

LABEL_NAMES = input_data.prepare_words_list(WANTED_WORDS.split(','))

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)
print("Clases:", LABEL_NAMES)

## E. Evaluación genérica TFLite

La función detecta automáticamente si la entrada es Float32, INT8 o INT16. Además
de accuracy, calcula precision, recall, F1 macro, latencia media, mediana y P95.

In [ ]:
def quantize_tensor(x_float, tensor_detail):
    dtype = tensor_detail["dtype"]

    if np.issubdtype(dtype, np.floating):
        return x_float.astype(dtype)

    scale, zero_point = tensor_detail["quantization"]

    if scale == 0:
        raise ValueError(
            f"Escala de cuantización inválida para {tensor_detail['name']}"
        )

    q = np.round(x_float / scale + zero_point)
    limits = np.iinfo(dtype)
    q = np.clip(q, limits.min, limits.max)

    return q.astype(dtype)


def dequantize_tensor(x_quant, tensor_detail):
    dtype = tensor_detail["dtype"]

    if np.issubdtype(dtype, np.floating):
        return x_quant.astype(np.float32)

    scale, zero_point = tensor_detail["quantization"]

    if scale == 0:
        return x_quant.astype(np.float32)

    return (x_quant.astype(np.float32) - zero_point) * scale


def evaluate_tflite_model(
    model_path,
    X,
    y,
    warmup=30,
    max_samples=None
):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]

    n_samples = len(X) if max_samples is None else min(len(X), max_samples)

    # Warm-up fuera de la medición.
    warmup_input = quantize_tensor(X[0], input_detail)
    for _ in range(warmup):
        interpreter.set_tensor(input_detail["index"], warmup_input)
        interpreter.invoke()

    predictions = []
    latencies_ms = []

    for i in range(n_samples):
        model_input = quantize_tensor(X[i], input_detail)

        start_ns = time.perf_counter_ns()
        interpreter.set_tensor(input_detail["index"], model_input)
        interpreter.invoke()
        output = interpreter.get_tensor(output_detail["index"])
        elapsed_ms = (time.perf_counter_ns() - start_ns) / 1e6

        output_float = dequantize_tensor(output, output_detail)
        predicted_class = int(np.argmax(output_float[0]))

        predictions.append(predicted_class)
        latencies_ms.append(elapsed_ms)

    y_eval = y[:n_samples]
    predictions = np.asarray(predictions)
    latencies_ms = np.asarray(latencies_ms)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_eval,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(y_eval, predictions),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
        "latency_mean_ms": float(np.mean(latencies_ms)),
        "latency_median_ms": float(np.median(latencies_ms)),
        "latency_p95_ms": float(np.percentile(latencies_ms, 95)),
        "input_dtype": np.dtype(input_detail["dtype"]).name,
        "output_dtype": np.dtype(output_detail["dtype"]).name,
        "predictions": predictions,
        "y_true": y_eval,
        "latencies_ms": latencies_ms,
    }

## F. Evaluación de los cuatro modelos

In [ ]:
MODEL_PATHS = {
    "Float32": FLOAT_MODEL_TFLITE,
    "PTQ INT8": PTQ_INT8_TFLITE,
    "QAT INT8": QAT_INT8_TFLITE,
    "PTQ INT16x8": PTQ_INT16X8_TFLITE,
}

for model_name, model_path in MODEL_PATHS.items():
    if not os.path.exists(model_path):
        raise FileNotFoundError(
            f"No existe {model_name}: {model_path}. "
            "Ejecute primero su celda de conversión."
        )

results = {}

for model_name, model_path in MODEL_PATHS.items():
    print(f"Evaluando {model_name}...")
    results[model_name] = evaluate_tflite_model(
        model_path,
        X_test,
        y_test,
        warmup=30
    )

    print(
        f"  Accuracy={results[model_name]['accuracy']:.4f}, "
        f"F1={results[model_name]['f1_macro']:.4f}, "
        f"latencia={results[model_name]['latency_mean_ms']:.4f} ms"
    )

## G. Tabla comparativa de precisión, memoria y latencia

El porcentaje de compresión se calcula respecto del archivo TFLite Float32.
La latencia corresponde al runtime de Colab; posteriormente debe repetirse el
benchmark en STM32N6570-DK.

In [ ]:
def gzip_file_size(path):
    with open(path, "rb") as f:
        return len(gzip.compress(f.read()))


float_size = os.path.getsize(FLOAT_MODEL_TFLITE)
rows = []

for model_name, model_path in MODEL_PATHS.items():
    ev = results[model_name]
    size_bytes = os.path.getsize(model_path)

    rows.append({
        "Modelo": model_name,
        "Accuracy": ev["accuracy"],
        "Precision_macro": ev["precision_macro"],
        "Recall_macro": ev["recall_macro"],
        "F1_macro": ev["f1_macro"],
        "TFLite_Bytes": size_bytes,
        "GZIP_Bytes": gzip_file_size(model_path),
        "Compresion_%": 100.0 * (1.0 - size_bytes / float_size),
        "Latencia_media_ms": ev["latency_mean_ms"],
        "Latencia_mediana_ms": ev["latency_median_ms"],
        "Latencia_P95_ms": ev["latency_p95_ms"],
        "Input_dtype": ev["input_dtype"],
        "Output_dtype": ev["output_dtype"],
    })

results_df = pd.DataFrame(rows)
results_df.to_csv(
    "comparacion_keyword_spotting_4_modelos.csv",
    index=False
)

display(results_df)

## H. Gráficos comparativos

In [ ]:
def plot_metric(column, ylabel, title, decimals=4):
    plt.figure(figsize=(8, 4.5))
    bars = plt.bar(results_df["Modelo"], results_df[column])

    for bar, value in zip(bars, results_df[column]):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.{decimals}f}",
            ha="center",
            va="bottom"
        )

    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        title.lower().replace(" ", "_") + ".png",
        dpi=200,
        bbox_inches="tight"
    )
    plt.show()


plot_metric("Accuracy", "Accuracy", "Accuracy por modelo")
plot_metric("F1_macro", "F1 macro", "F1 macro por modelo")
plot_metric("TFLite_Bytes", "Bytes", "Tamaño TFLite por modelo", decimals=0)
plot_metric(
    "Compresion_%",
    "Compresión [%]",
    "Compresión respecto de Float32",
    decimals=2
)
plot_metric(
    "Latencia_media_ms",
    "ms/inferencia",
    "Latencia media en Colab",
    decimals=4
)
plot_metric(
    "Latencia_P95_ms",
    "ms/inferencia",
    "Latencia P95 en Colab",
    decimals=4
)

## I. Matrices de confusión normalizadas

In [ ]:
def plot_confusion_matrix(model_name, y_true, y_pred):
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(len(LABEL_NAMES)),
        normalize="true"
    )

    plt.figure(figsize=(9, 7))
    plt.imshow(cm, interpolation="nearest", aspect="auto")
    plt.title(f"Matriz de confusión normalizada — {model_name}")
    plt.colorbar(label="Proporción")

    ticks = np.arange(len(LABEL_NAMES))
    plt.xticks(ticks, LABEL_NAMES, rotation=45, ha="right")
    plt.yticks(ticks, LABEL_NAMES)
    plt.xlabel("Predicción")
    plt.ylabel("Clase real")
    plt.tight_layout()

    filename = (
        "confusion_"
        + model_name.lower().replace(" ", "_").replace("x", "x")
        + ".png"
    )
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()


for model_name, ev in results.items():
    plot_confusion_matrix(
        model_name,
        ev["y_true"],
        ev["predictions"]
    )